In [ ]:
# ============================================================
# CodStaticDataset.ipynb
# ============================================================
# INPUT:  panel_{YEAR}.csv (output di CodPerformancePanel)
# OUTPUT: static_{YEAR}.csv
#   - una riga per loan
#   - covariate statiche a t=0 (prima riga disponibile per loan)
#   - first_default_age: primo mese di delinquency su tutta la storia
#   - default_12m: 1 se first_default_age <= HORIZON, 0 altrimenti
#     (qualsiasi delinquency != 0, coerente con is_default() del modello)
#
# STRUTTURA DRIVE:
#   MyDrive/thesis_data/output/panel_{YEAR}.csv   <- input
#   MyDrive/thesis_data/output/static_{YEAR}.csv  <- output
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive montato:', os.path.exists('/content/drive/MyDrive'))

In [ ]:
# ============================================================
# CELLA 1 - Configurazione e definizione colonne
# ============================================================
# STATIC_COLS e' definito qui perche' serve gia' in cella 2
# per il filtro usecols al caricamento.
# MODIFICA SOLO DRIVE_ROOT e YEAR.

import os, gc
import pandas as pd
import numpy as np

# ---- UNICI DUE VALORI DA MODIFICARE ----
DRIVE_ROOT = '/content/drive/MyDrive/thesis_data'
YEAR       = 2022
# ----------------------------------------

OUTPUT_DIR  = os.path.join(DRIVE_ROOT, 'output')
PANEL_PATH  = os.path.join(OUTPUT_DIR, f'panel_{YEAR}.csv')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f'static_{YEAR}.csv')
HORIZON     = 12

STATIC_COLS = [
    'loan_sequence_number',
    # --- Origination Freddie ---
    'credit_score', 'first_time_homebuyer',
    'original_ltv', 'original_cltv', 'original_dti',
    'interest_rate', 'loan_term', 'loan_purpose_orig',
    'property_type', 'occupancy_status_orig',
    'num_units', 'num_borrowers', 'state_code',
    'channel', 'amortization_type',
    'loan_amount', 'property_value', 'mi_pct', 'loan_amount_r',
    # --- Fairness (HMDA) ---
    'derived_race', 'applicant_race_1',
    'derived_sex', 'applicant_sex',
    'derived_ethnicity', 'applicant_ethnicity_1',
    'applicant_age', 'applicant_age_above_62',
    'co_applicant_age', 'co_applicant_age_above_62',
    'income',
    # --- Geo ---
    'msa', 'postal_code', 'county_code', 'census_tract',
    'tract_minority_population_percent',
    'ffiec_msa_md_median_family_income',
    'tract_to_msa_income_percentage',
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Anno             : {YEAR}')
print(f'Input panel      : {PANEL_PATH}')
print(f'Output static    : {OUTPUT_PATH}')
print(f'Orizzonte mesi   : {HORIZON}')
print(f'Colonne statiche : {len(STATIC_COLS)}')
print()
status = 'OK' if os.path.exists(PANEL_PATH) else 'ERRORE - non trovato'
print(f'  {status}  panel CSV: {PANEL_PATH}')

In [ ]:
# ============================================================
# CELLA 2 - Carica panel (ottimizzato RAM)
# ============================================================
# Legge solo le colonne necessarie tramite usecols:
#   - per la label  : loan_sequence_number, loan_age, current_loan_delinquency_status
#   - per t=0       : tutte le STATIC_COLS
# Le centinaia di colonne extra del panel non vengono caricate.

LABEL_COLS = ['loan_sequence_number', 'loan_age', 'current_loan_delinquency_status']
T0_COLS    = ['loan_sequence_number', 'loan_age'] + [
    c for c in STATIC_COLS if c != 'loan_sequence_number'
]
COLS_TO_READ = list(dict.fromkeys(LABEL_COLS + T0_COLS))  # unione, no duplicati

# Verifica quali colonne esistono nel file
header = pd.read_csv(PANEL_PATH, nrows=0)
cols_in_file = [c for c in COLS_TO_READ if c in header.columns]
cols_missing = [c for c in COLS_TO_READ if c not in header.columns]
del header

if cols_missing:
    print(f'Colonne non trovate nel file ({len(cols_missing)}): {cols_missing}')

print(f'Caricamento panel ({len(cols_in_file)} colonne su {len(COLS_TO_READ)} richieste)...')

panel = pd.read_csv(
    PANEL_PATH,
    usecols=cols_in_file,
    dtype=str,
    low_memory=False
)

panel['loan_age'] = pd.to_numeric(panel['loan_age'], errors='coerce')

# Colonne statiche effettivamente disponibili nel panel caricato
STATIC_COLS_AVAIL = [c for c in STATIC_COLS if c in panel.columns]

gc.collect()
print(f'Panel caricato              : {len(panel):,} righe x {panel.shape[1]} colonne')
print(f'Loan unici                  : {panel["loan_sequence_number"].nunique():,}')
print(f'Colonne statiche disponibili: {len(STATIC_COLS_AVAIL)}/{len(STATIC_COLS)}')

In [ ]:
# ============================================================
# CELLA 3 - Costruisci label default_12m
# ============================================================
# Default = qualsiasi delinquency (current_loan_delinquency_status != 0)
# Implementazione vettorializzata — evita apply() riga per riga.

delq     = pd.to_numeric(panel['current_loan_delinquency_status'], errors='coerce')
delq_str = panel['current_loan_delinquency_status'].astype(str).str.strip()

panel['_is_default'] = np.where(
    delq.notna(),
    (delq != 0).astype(int),                       # caso numerico
    (~delq_str.isin({'', '0', '00'})).astype(int)  # caso stringa non parsabile
)

del delq, delq_str
gc.collect()

# Primo mese di default per loan su tutta la storia
first_default_age = (
    panel[panel['_is_default'] == 1]
    .groupby('loan_sequence_number')['loan_age']
    .min()
    .reset_index()
    .rename(columns={'loan_age': 'first_default_age'})
)

# Tutti i loan unici del panel
all_loans = pd.DataFrame(
    panel['loan_sequence_number'].unique(),
    columns=['loan_sequence_number']
)

# Left join: chi non ha mai defaultato → first_default_age = NaN
all_loans = all_loans.merge(first_default_age, on='loan_sequence_number', how='left')

# Label: 1 se primo default entro HORIZON mesi
all_loans['default_12m'] = np.where(
    all_loans['first_default_age'].notna() &
    (all_loans['first_default_age'] <= HORIZON),
    1, 0
)

label = all_loans[['loan_sequence_number', 'default_12m', 'first_default_age']]

panel.drop(columns=['_is_default'], inplace=True)
gc.collect()

print(f'Loan totali nel panel      : {len(label):,}')
print(f'Default (label=1)          : {label["default_12m"].sum():,} ({label["default_12m"].mean()*100:.2f}%)')
print(f'Non default (label=0)      : {(label["default_12m"]==0).sum():,}')

In [ ]:
# ============================================================
# CELLA 4 - Estrai covariate a t=0 e libera panel
# ============================================================

t0 = (
    panel.sort_values(['loan_sequence_number', 'loan_age'])
    .groupby('loan_sequence_number', as_index=False)
    .first()
)[STATIC_COLS_AVAIL].copy()

del panel
gc.collect()

print(f'Righe t=0 estratte: {len(t0):,}')

In [ ]:
# ============================================================
# CELLA 5 - Join t=0 con label e salvataggio
# ============================================================

static = t0.merge(label, on='loan_sequence_number', how='inner')

front = ['loan_sequence_number', 'default_12m', 'first_default_age']
rest  = [c for c in static.columns if c not in front]
static = static[front + rest]

static.to_csv(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6

print(f'Salvato          : {OUTPUT_PATH}')
print(f'Dimensione       : {size_mb:.1f} MB')
print(f'Righe            : {len(static):,}')
print(f'Colonne          : {len(static.columns)}')
print()
print(f'Default rate     : {static["default_12m"].mean()*100:.2f}%')
print()
print('--- DISTRIBUZIONE LABEL ---')
print(static['default_12m'].value_counts())
print()
static.head(3)

In [ ]:
# ============================================================
# CELLA 6 - Sanity check
# ============================================================

print('=== SANITY CHECK ===')
print()

# 1. Nessun duplicato
dups = static.duplicated(subset='loan_sequence_number').sum()
print(f'Duplicati su loan_sequence_number: {dups}',
      '\u2713' if dups == 0 else '\u26a0 ATTENZIONE')

# 2. Missing rate colonne chiave
key_cols = ['credit_score', 'original_ltv', 'original_dti',
            'interest_rate', 'derived_race', 'derived_sex', 'default_12m']
print()
print('Missing rate colonne chiave:')
for c in [c for c in key_cols if c in static.columns]:
    miss = static[c].isna().mean() * 100
    print(f'  {c:<45} {miss:.1f}%')

# 3. Distribuzione race
if 'derived_race' in static.columns:
    print()
    print('Distribuzione derived_race:')
    print(static['derived_race'].value_counts(normalize=True).mul(100).round(1).to_string())

# 4. Default rate per race
if 'derived_race' in static.columns:
    print()
    print('Default rate per derived_race:')
    print(static.groupby('derived_race')['default_12m'].mean().mul(100).round(2).to_string())